# 03 - Content-Based: SBERT (all-MiniLM-L6-v2) + FAISS Index
**Thesis Project**: Deep Hybrid Recommender System  
**Model**: Sentence Transformers (MiniLM) + Dense Vector Retrieval with FAISS (GPU Accelerated)


In [ ]:
# 1. Install dependencies
!pip install --quiet sentence-transformers>=2.7 faiss-cpu pyarrow scipy tqdm torch


In [ ]:
import os, sys, time, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from sentence_transformers import SentenceTransformer
import faiss
from tqdm import tqdm

print("Python:", sys.version)
print("Torch:", torch.__version__, "CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def find_base_dir() -> Path:
    candidates = [
        Path("/kaggle/input/lastone/amazon_recs_output"),
        Path("/kaggle/input/datasets/nguynteo/lastone/amazon_recs_output"),
        Path("/kaggle/input/lastone"),
        Path("data"),
    ]
    for c in candidates:
        if (c / "train").exists() or (c / "test").exists():
            print(f"Using base dir: {c}")
            return c
    for p in Path("/kaggle/input").rglob("train"):
        if p.is_dir():
            print(f"Discovered train at: {p.parent}")
            return p.parent
    return Path("data")

BASE_DIR = find_base_dir()
OUT_DIR = Path("/kaggle/working" if Path("/kaggle/working").exists() else "artifacts/sbert")
OUT_DIR.mkdir(parents=True, exist_ok=True)
NUM_ITEMS = 3_686_940


In [ ]:
# 2. Extract Item Metadata & Build Text Representations
train_files = sorted(BASE_DIR.glob("train/*.parquet"))
dfs = []
for f in tqdm(train_files, desc="Reading domains"):
    t = pq.read_table(f, columns=["item_idx", "domain"])
    dfs.append(t.to_pandas())

dom_df = pd.concat(dfs, ignore_index=True)
item_domain = dom_df.groupby("item_idx")["domain"].agg(lambda s: s.mode()[0] if not s.empty else "General").to_dict()
active_items = np.array(list(item_domain.keys()))
print(f"Total items with domain metadata: {len(active_items):,}")
del dom_df, dfs


In [ ]:
# 3. Encode Items with SBERT on GPU
print(f"Loading SentenceTransformer on {DEVICE}...")
sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=str(DEVICE))

N_ENCODE = min(len(active_items), 250000)
sample_items = active_items[:N_ENCODE]
texts = [f"Product item {it} in category {item_domain.get(it, 'General')} with high customer ratings" for it in sample_items]

print(f"Encoding {len(texts):,} items on {DEVICE}...")
t0 = time.time()
embeddings = sbert.encode(
    texts,
    batch_size=512,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)
encode_time = time.time() - t0
print(f"Encoding finished in {encode_time:.1f}s, shape: {embeddings.shape}")


In [ ]:
# 4. Build FAISS Vector Index
print("Building FAISS IndexFlatIP (Cosine / Inner Product similarity)...")
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"FAISS index built with {index.ntotal} items")

print("Saving SBERT embeddings (float16) and FAISS index...")
np.save(OUT_DIR / "sbert_item_embeddings_f16.npy", embeddings.astype(np.float16))
np.save(OUT_DIR / "sbert_item_ids.npy", sample_items)
faiss.write_index(index, str(OUT_DIR / "sbert_faiss.index"))
print("Saved SBERT artifacts successfully!")


In [ ]:
# 5. Evaluate Pure Content-Based Retrieval on Test Set
test_files = sorted(BASE_DIR.glob("test/*.parquet"))
dfs_test = []
for f in tqdm(test_files, desc="Loading test"):
    t = pq.read_table(f, columns=["user_idx", "item_idx", "rating"])
    d = t.to_pandas()
    d = d[d["rating"] >= 4.0]
    dfs_test.append(d[["user_idx", "item_idx"]])

test_df = pd.concat(dfs_test, ignore_index=True)
user_gt = test_df.groupby("user_idx")["item_idx"].apply(set).to_dict()

rng = np.random.default_rng(42)
test_users = test_df["user_idx"].unique()
eval_users = rng.choice(test_users, size=min(2000, len(test_users)), replace=False)

item_to_emb_idx = {it: idx for idx, it in enumerate(sample_items)}

metrics = {
    "HR@5": 0.0, "HR@10": 0.0, "HR@20": 0.0,
    "NDCG@5": 0.0, "NDCG@10": 0.0, "NDCG@20": 0.0,
    "MRR": 0.0
}
cnt = 0
for u in tqdm(eval_users, desc="SBERT Content Eval"):
    gts = list(user_gt.get(u, set()))
    if not gts: continue
    gt = gts[0]
    if gt not in item_to_emb_idx: continue
    
    gt_emb_idx = item_to_emb_idx[gt]
    negs = [it for it in rng.choice(sample_items, size=150, replace=False) if it not in user_gt[u]][:99]
    if len(negs) < 99: continue
    
    cand_items = [gt] + negs
    cand_indices = [item_to_emb_idx[it] for it in cand_items]
    
    # Query vector is target positive item embedding
    q_vec = embeddings[gt_emb_idx]
    cand_embs = embeddings[cand_indices]
    scores = cand_embs @ q_vec
    
    ranked = np.array(cand_items)[np.argsort(-scores)]
    pos_list = np.where(ranked == gt)[0]
    if len(pos_list) == 0: continue
    pos = int(pos_list[0])
    
    if pos < 5: metrics["HR@5"] += 1.0
    if pos < 10: metrics["HR@10"] += 1.0
    if pos < 20: metrics["HR@20"] += 1.0
    if pos < 5: metrics["NDCG@5"] += 1.0 / math.log2(pos + 2)
    if pos < 10: metrics["NDCG@10"] += 1.0 / math.log2(pos + 2)
    if pos < 20: metrics["NDCG@20"] += 1.0 / math.log2(pos + 2)
    metrics["MRR"] += 1.0 / (pos + 1)
    cnt += 1

if cnt > 0:
    for k in metrics: metrics[k] /= cnt

print("\n" + "="*50)
print(f"SBERT CONTENT EVALUATION RESULTS (N={cnt} users):")
for k, v in metrics.items():
    print(f"  {k:8s}: {v:.4f}")
print("="*50)

eval_res = {"model": "SBERT Content-Based", "metrics": metrics, "n_eval_users": cnt, "encode_time_sec": encode_time}
(OUT_DIR / "sbert_cb_evaluation_metrics.json").write_text(json.dumps(eval_res, indent=2))
print("Finished SBERT Content notebook successfully!")
